# Multi-Asset Portfolio Optimization

Quantitative portfolio construction, constrained Sharpe-ratio optimization, risk analytics, ESG allocation and crypto diversification.


## Project Context

This project was carried out independently by **Vincent Haïk Karakoseian** in collaboration with **Grégoire Hug, CEO of WeeFin**.

The quantitative analysis, portfolio construction, optimization framework, risk metrics, asset selection and Python implementation presented in this repository were developed by Vincent Haïk Karakoseian.

> This notebook is presented as an academic portfolio project. Historical and in-sample results are not forecasts of future performance.


## Part I — Baseline Multi-Asset Portfolio

The first portfolio combines global equities, investment-grade bonds, gold and Bitcoin. The objective is to maximize the Sharpe ratio under long-only, volatility and crypto-exposure constraints.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Definition of Yahoo Finance tickers for each asset in the first portfolio
tickers = [
    "URTH",     # Global equities (MSCI World)
    "BNDX",     # International investment-grade sovereign bonds
    "LQD",      # USD investment-grade corporate bonds
    "GLD",      # Gold
    "BTC-USD"   # Bitcoin
]

from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def download_if_colab(path):
    """Download a generated file when running in Google Colab; do nothing locally."""
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        pass


In [ ]:
# 1) Price download (Yahoo Finance) + data cleaning

# Retrieve daily prices over 5 years (2020 to end of 2025)
start_date = "2020-01-01"
end_date = "2025-12-26"

raw = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=True,   # adjusted prices (splits/dividends)
    group_by="column"   # more stable column structure
)

# Robust extraction of prices
if isinstance(raw.columns, pd.MultiIndex):
    # Typical format: (PriceField, Ticker)
    if "Close" in raw.columns.get_level_values(0):
        prices = raw["Close"]
    elif "Adj Close" in raw.columns.get_level_values(0):
        prices = raw["Adj Close"]
    else:
        raise KeyError(f"Neither 'Close' nor 'Adj Close' found in columns: {raw.columns.levels[0]}")
else:
    prices = raw["Close"] if "Close" in raw.columns else raw["Adj Close"]

# Data cleaning
prices = prices.dropna(how="all")          # remove fully empty days
prices = prices.ffill(limit=2)             # forward-fill up to 2 days (useful: closed markets vs crypto)
prices = prices.dropna(axis=1, how="all")  # remove tickers that are 100% empty (safety check)
prices = prices.dropna()                   # align on remaining common dates


# 2) Returns + correlation matrix

returns_daily = prices.pct_change().dropna()
corr_matrix = returns_daily.corr()


# 3) Excel export with multiple sheets

output_file = OUTPUT_DIR / "prices_and_correlation.xlsx"

with pd.ExcelWriter(output_file) as writer:
    prices.to_excel(writer, sheet_name="Prices_cleaned")
    returns_daily.to_excel(writer, sheet_name="Returns_daily")
    corr_matrix.to_excel(writer, sheet_name="Correlation_matrix")

# Automatic download (Google Colab)
download_if_colab(output_file)

[*********************100%***********************]  5 of 5 completed


In [ ]:
# Align the data by removing days with missing values for at least one asset

prices = prices.dropna(how="any")  # remove dates where at least one value is missing

# Compute daily returns (log or simple returns could be used – here: simple returns)
returns_daily = prices.pct_change().dropna()

# Risk-free rate (rf) for the Sharpe ratio
# Reasonable choice for the euro area: use €STR (Euro Short-Term Rate) as a proxy
# for the risk-free rate, as it is an overnight rate published daily by the ECB
# and commonly used as a reference risk-free rate.
# Source (ECB): €STR definition and methodology:
# - https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_short-term_rate/html/index.en.html
# - https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_short-term_rate/html/eurostr_overview.en.html
#
#  example: €STR = 1.925% (23/12/2025)
rf_annual = 0.01925  # 1.925% annual

# Conversion to a daily rate (consistent with daily returns)
annual_factor = 252  # number of trading days per year (~252)
rf_daily = (1 + rf_annual) ** (1 / annual_factor) - 1

# Compute annualized performance metrics
mean_daily = returns_daily.mean()
vol_daily = returns_daily.std()

# Annualization
mean_annual = mean_daily * annual_factor
vol_annual = vol_daily * np.sqrt(annual_factor)

# Sharpe ratio based on excess returns
mean_annual_excess = (mean_daily - rf_daily) * annual_factor
sharpe_ratio = mean_annual_excess / vol_annual

# Compile metrics into a DataFrame
metrics = pd.DataFrame({
    "Annual_return_%": mean_annual * 100,
    "Annual_excess_return_%": mean_annual_excess * 100,
    "Annual_volatility_%": vol_annual * 100,
    "Sharpe_ratio": sharpe_ratio
}).sort_values("Sharpe_ratio", ascending=False)

# Export to Excel
output_file = OUTPUT_DIR / "metrics.xlsx"

with pd.ExcelWriter(output_file) as writer:
    metrics.to_excel(writer, sheet_name="Metrics")

# download
download_if_colab(output_file)

In [ ]:
from scipy.optimize import minimize

TRADING_DAYS = 252
crypto_tickers = ["BTC-USD", "ETH-USD", "ADA-USD", "SOL-USD", "DOT-USD", "HBAR-USD", "MATIC-USD", "LINK-USD"]
rf = 0.01925  # annual risk-free rate

returns_daily = prices.pct_change().dropna()

mu = returns_daily.mean() * TRADING_DAYS        # annualized expected returns
cov = returns_daily.cov() * TRADING_DAYS        # annualized covariance matrix

assets = returns_daily.columns.tolist()
n = len(assets)

# Indices of crypto assets in the asset universe
crypto_idx = np.array([assets.index(t) for t in crypto_tickers if t in assets], dtype=int)

cov_mat = cov.values
def port_return(w):
    """Portfolio expected annual return."""
    return float(np.dot(w, mu.values))

def port_vol(w):
    """Portfolio annualized volatility."""
    return float(np.sqrt(np.dot(w, np.dot(cov_mat, w))))

def neg_sharpe(w):
    """Negative Sharpe ratio (objective function to minimize)."""
    v = port_vol(w)
    if v <= 1e-12:
        return 1e6
    return -((port_return(w) - rf) / v)

# Long-only portfolio bounds
bounds = [(0.0, 1.0) for _ in range(n)]

def run_optim(vol_max, crypto_max, w0=None, maxiter=400, disp=False):
    """
    Sharpe ratio maximization under constraints.

    Constraints:
    - Fully invested portfolio (sum of weights = 1)
    - Maximum portfolio volatility
    - Maximum total weight allocated to cryptocurrencies

    Returns:
        Dictionary containing optimization success flag, message,
        optimal weights, and portfolio metrics.
    """

    constraints = [
        {"type": "eq",   "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: vol_max - port_vol(w)}
    ]

    if len(crypto_idx) > 0:
        constraints.append({
            "type": "ineq",
            "fun": lambda w: crypto_max - np.sum(w[crypto_idx])
        })

    # Initial guess
    if w0 is None:
        w0 = np.ones(n) / n

    res = minimize(
        neg_sharpe,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": maxiter, "ftol": 1e-10, "disp": disp}
    )

    out = {
        "success": bool(res.success),
        "message": str(res.message),
        "sharpe": np.nan,
        "ret": np.nan,
        "vol": np.nan,
        "crypto_weight": np.nan,
        "n_active": np.nan,
        "w_opt": None
    }

    if res.success:
        w = res.x
        r = port_return(w)
        v = port_vol(w)
        s = (r - rf) / v if v > 1e-12 else np.nan
        cw = float(np.sum(w[crypto_idx])) if len(crypto_idx) > 0 else 0.0

        out.update({
            "sharpe": s,
            "ret": r,
            "vol": v,
            "crypto_weight": cw,
            "n_active": int(np.sum(w > 1e-4)),
            "w_opt": w
        })

    return out


# Constraint grids
vol_grid = np.arange(0.00, 0.30 + 1e-12, 0.01)    # 0% to 30% volatility, 1% step
crypto_grid = np.arange(0.00, 0.50 + 1e-12, 0.01) # 0% to 50% crypto exposure, 1% step

results = []
w0 = np.ones(n) / n

last_w = w0.copy()

for vol_max in vol_grid:
    for crypto_max in crypto_grid:
        out = run_optim(vol_max, crypto_max, w0=last_w, maxiter=300, disp=False)

        results.append({
            "vol_max": vol_max,
            "crypto_max": crypto_max,
            "success": out["success"],
            "message": out["message"],
            "sharpe": out["sharpe"],
            "ret": out["ret"],
            "vol": out["vol"],
            "crypto_weight": out["crypto_weight"],
            "n_active": out["n_active"]
        })
#If the optimization was successful, we keep the solution found and use it as the starting point for the next iteration.
        if out["success"] and out["w_opt"] is not None:
            last_w = out["w_opt"]

grid_df = pd.DataFrame(results)

In [ ]:
# Sort results to identify the best-performing combinations
grid_ok = grid_df[grid_df["success"]].sort_values("sharpe", ascending=False)

# Prepare a clean export version with percentage formatting
grid_ok_export = grid_ok.copy()
grid_ok_export["vol_max_%"] = (grid_ok_export["vol_max"] * 100).round(1)
grid_ok_export["crypto_max_%"] = (grid_ok_export["crypto_max"] * 100).round(0)
grid_ok_export["ret_%"] = (grid_ok_export["ret"] * 100).round(2)
grid_ok_export["vol_%"] = (grid_ok_export["vol"] * 100).round(2)
grid_ok_export["crypto_weight_%"] = (grid_ok_export["crypto_weight"] * 100).round(2)

# Columns to highlight in the summary sheet
cols = [
    "vol_max_%", "crypto_max_%",
    "sharpe", "ret_%", "vol_%", "crypto_weight_%", "n_active"
]

# Export results to Excel
with pd.ExcelWriter(
    "grid_optimisation_results_clean.xlsx",
    engine="xlsxwriter"
) as writer:
    grid_ok_export[cols].to_excel(writer, sheet_name="Summary", index=False)
    grid_ok_export.to_excel(writer, sheet_name="Full", index=False)

files.download("grid_optimisation_results_clean.xlsx")

In [ ]:
# Identify the maximum Sharpe ratio
best_sharpe = grid_ok["sharpe"].max()

# Select all portfolios achieving (approximately) the maximum Sharpe ratio
candidates = grid_ok[np.isclose(grid_ok["sharpe"], best_sharpe)]

# Conservative selection rule:
# choose the portfolio with the lowest volatility cap,
# then (if needed) the lowest crypto exposure cap
best = candidates.sort_values(
    ["vol_max", "crypto_max"],
    ascending=[True, True]
).iloc[0]

best

,783
vol_max,0.15
crypto_max,0.18
success,True
message,Optimization terminated successfully
sharpe,1.097259
ret,0.183527
vol,0.149716
crypto_weight,0.179681
n_active,3.0


In [ ]:
# Selected constraints from the best Sharpe configuration
chosen_vol_max = best["vol_max"]
chosen_crypto_max = best["crypto_max"]

# Run the optimization again with the optimal constraints
out = run_optim(
    chosen_vol_max,
    chosen_crypto_max,
    w0=np.ones(n) / n,
    maxiter=3000,
    disp=True
)

# Extract optimal results
w_opt = out["w_opt"]
ret_opt = out["ret"]
vol_opt = out["vol"]
sharpe_opt = out["sharpe"]
crypto_weight = out["crypto_weight"]

# Build and clean the allocation vector
alloc = pd.Series(w_opt, index=assets).sort_values(ascending=False)
alloc = alloc[alloc > 1e-4]

# Display results
print("\n--- Optimized portfolio (maximum Sharpe ratio) ---")
print(f"Estimated annual return : {ret_opt:.2%}")
print(f"Annual volatility       : {vol_opt:.2%}  (Volatility constraint: {chosen_vol_max:.0%})")
print(f"Sharpe ratio            : {sharpe_opt:.3f}  (rf = {rf:.2%})")
print(f"Total crypto exposure   : {crypto_weight:.2%}  (Crypto constraint: {chosen_crypto_max:.0%})")

print("\n--- Optimal asset weights (filtered) ---")
print((alloc * 100).round(2))

In [ ]:
# 1) Build returns of the optimal portfolio

# Align w_opt with the column order of the returns DataFrame
assets = returns_daily.columns.tolist()
w_series = pd.Series(w_opt, index=assets)

# Daily portfolio returns
port_ret_daily = (returns_daily.mul(w_series, axis=1)).sum(axis=1)

# 2) Annual KPIs (annualization)
annual_factor = 252
rf_annual = rf

port_ret_annual = port_ret_daily.mean() * annual_factor
port_vol_annual = port_ret_daily.std() * np.sqrt(annual_factor)
port_sharpe = (port_ret_annual - rf_annual) / port_vol_annual

# 3) Historical VaR / CVaR (daily)
alpha = 0.95

losses = -port_ret_daily
VaR_95 = np.percentile(losses, alpha * 100)        # 95% VaR (loss)
CVaR_95 = losses[losses >= VaR_95].mean()          # Expected Shortfall

# 4) Drawdown analysis
cum = (1 + port_ret_daily).cumprod()
running_max = cum.cummax()
drawdown = cum / running_max - 1
max_drawdown = drawdown.min()

# 5) Tables to export

# A) KPI summary
kpi = pd.DataFrame({
    "Metric": [
        "Annual return",
        "Annual volatility",
        "Sharpe ratio",
        "VaR 95% (daily)",
        "CVaR 95% (daily)",
        "Maximum drawdown"
    ],
    "Value": [
        port_ret_annual,
        port_vol_annual,
        port_sharpe,
        VaR_95,
        CVaR_95,
        max_drawdown
    ]
})

# Percentage formatting for readability
kpi["Value_%"] = np.nan
kpi.loc[
    kpi["Metric"].isin([
        "Annual return",
        "Annual volatility",
        "VaR 95% (daily)",
        "CVaR 95% (daily)",
        "Maximum drawdown"
    ]),
    "Value_%"
] = kpi["Value"] * 100

# B) Portfolio composition (weights)
alloc_full = pd.DataFrame({
    "Asset": assets,
    "Weight": w_series.values
}).sort_values("Weight", ascending=False)

alloc_full["Weight_%"] = alloc_full["Weight"] * 100

# C) Portfolio return and drawdown time series
ts = pd.DataFrame({
    "Portfolio_return_daily": port_ret_daily,
    "Cumulative_value": cum,
    "Drawdown": drawdown
})

# 6) Excel export + download
output_file = "optimal_portfolio_report.xlsx"

with pd.ExcelWriter(output_file) as writer:
    # Sheet 1: KPIs
    kpi.to_excel(writer, sheet_name="KPI", index=False)

    # Sheet 2: Allocation
    alloc_full.to_excel(writer, sheet_name="Allocation", index=False)

    # Sheet 3: Time series
    ts.to_excel(writer, sheet_name="Timeseries", index=True)

print("Excel file created:", output_file)

from google.colab import files
files.download(output_file)

## Part II — ESG / PEA / Crypto-Thematic Portfolio

The second portfolio uses a 40% ETF / 40% European equities / 20% crypto top-down allocation, then optimizes weights within each asset class under volatility constraints.


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

# Definition of Yahoo Finance tickers for each asset in the second portfolio

tickers_ETF = [
    "BGRN",       # USD green bonds ETF (replacement for LQD)
    "EHYB.MI",    # EUR high-yield corporate bond ESG ETF
    "PUST.PA",    # Amundi PEA Nasdaq-100 ETF (US technology)
    "MFED.PA",    # Amundi MSCI EMU ESG Broad ETF (Eurozone equities ESG) – proxy for PAEM
    "CW8.PA",     # Amundi MSCI World PEA ETF
    "EXV4.DE",    # iShares STOXX Europe 600 Health Care ETF (European healthcare) – proxy for STZ
    "EXH9.DE",    # iShares STOXX Europe 600 Utilities ETF (European utilities) – proxy for UTI
]

tickers_shares = [
    "ASML.AS",    # ASML Holding (Netherlands)
    "SAP.DE",     # SAP SE (Germany)
    "SU.PA",      # Schneider Electric (France)
    "IBE.MC",     # Iberdrola (Spain)
    "NOVO-B.CO",  # Novo Nordisk B (Denmark)
    "VIE.PA",     # Veolia Environnement (France)
]

tickers_cryptos = [
    "BTC-USD",    # Bitcoin (cryptocurrency)
    "ETH-USD",    # Ethereum (cryptocurrency)
    "ADA-USD",    # Cardano (cryptocurrency)
    "SOL-USD",    # Solana (cryptocurrency)
    "DOT-USD",    # Polkadot (cryptocurrency)
    "HBAR-USD",   # Hedera Hashgraph (cryptocurrency)
    "MATIC-USD",  # Polygon (cryptocurrency, additional altcoin)
    "LINK-USD"    # Chainlink (cryptocurrency, additional altcoin)
]

# All tickers grouped by asset class
tickers_all = [tickers_ETF, tickers_shares, tickers_cryptos]

In [3]:
# 1) Price download (Yahoo Finance) + data cleaning

for tickers in tickers_all:

    # Retrieve daily prices over 5 years (2020 to end of 2025)
    start_date = "2020-01-01"
    end_date = "2025-12-26"

    raw = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=True,   # adjusted prices (splits/dividends)
        group_by="column"   # more stable column structure
    )

    # Robust price extraction
    if isinstance(raw.columns, pd.MultiIndex):
        # Typical format: (PriceField, Ticker)
        if "Close" in raw.columns.get_level_values(0):
            prices = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            prices = raw["Adj Close"]
        else:
            raise KeyError(
                f"Neither 'Close' nor 'Adj Close' found in columns: {raw.columns.levels[0]}"
            )
    else:
        prices = raw["Close"] if "Close" in raw.columns else raw["Adj Close"]

    # Data cleaning
    prices = prices.dropna(how="all")          # remove fully empty days
    prices = prices.ffill(limit=2)             # forward-fill up to 2 days (useful: closed markets vs crypto)
    prices = prices.dropna(axis=1, how="all")  # remove tickers that are 100% empty (safety check)
    prices = prices.dropna()                   # align on remaining common dates


    # 2) Returns + correlation matrix

    returns_daily = prices.pct_change().dropna()
    corr_matrix = returns_daily.corr()


    # 3) Excel export

    if tickers[0] == "BGRN":
        output_file = OUTPUT_DIR / "prices_and_correlation_ETF.xlsx"
    elif tickers[0] == "ASML.AS":
        output_file = OUTPUT_DIR / "prices_and_correlation_shares.xlsx"
    else:
        output_file = OUTPUT_DIR / "prices_and_correlation_cryptos.xlsx"

    with pd.ExcelWriter(output_file) as writer:
        prices.to_excel(writer, sheet_name="Prices_cleaned")
        returns_daily.to_excel(writer, sheet_name="Returns_daily")
        corr_matrix.to_excel(writer, sheet_name="Correlation_matrix")

    # Automatic download
    download_if_colab(output_file)

[*********************100%***********************]  7 of 7 completed


[*********************100%***********************]  6 of 6 completed


[*********************100%***********************]  8 of 8 completed


In [4]:
prices_ETF = pd.read_excel(OUTPUT_DIR / "prices_and_correlation_ETF.xlsx")
prices_shares = pd.read_excel(OUTPUT_DIR / "prices_and_correlation_shares.xlsx")
prices_cryptos = pd.read_excel(OUTPUT_DIR / "prices_and_correlation_cryptos.xlsx")

prices_all = [prices_ETF, prices_shares, prices_cryptos]

for prices in prices_all:

    # Align the data by removing days with missing values for at least one asset

    prices = prices.dropna(how="any")  # remove dates where at least one value is missing

    prices["Date"] = pd.to_datetime(prices["Date"])
    prices = prices.set_index("Date")

    # Compute daily returns
    returns_daily = prices.pct_change().dropna()

    # Same as before we take €STER level at the end of 2025
    rf_annual = 0.01925  # 1.925% annual

    # Convert to a daily rate
    annual_factor = 252  # number of trading days per year
    rf_daily = (1 + rf_annual) ** (1 / annual_factor) - 1

    # Compute annual performance metrics
    mean_daily = returns_daily.mean()
    vol_daily = returns_daily.std()

    # Annualization
    mean_annual = mean_daily * annual_factor
    vol_annual = vol_daily * np.sqrt(annual_factor)

    # Sharpe ratio based on excess returns
    mean_annual_excess = (mean_daily - rf_daily) * annual_factor
    sharpe_ratio = mean_annual_excess / vol_annual

    # Compile metrics into a DataFrame
    metrics = pd.DataFrame({
        "Annual_return_%": mean_annual * 100,
        "Annual_excess_return_%": mean_annual_excess * 100,
        "Annual_volatility_%": vol_annual * 100,
        "Sharpe_ratio": sharpe_ratio
    }).sort_values("Sharpe_ratio", ascending=False)

    # Define output file name based on asset class
    if prices.columns[0] == "BGRN":
        output_file = OUTPUT_DIR / "metrics_ETF.xlsx"
    elif prices.columns[0] == "ASML.AS":
        output_file = OUTPUT_DIR / "metrics_shares.xlsx"
    else:
        output_file = OUTPUT_DIR / "metrics_cryptos.xlsx"

    with pd.ExcelWriter(output_file) as writer:
        metrics.to_excel(writer, sheet_name="Metrics")

    #download
    download_if_colab(output_file)

In [5]:
from scipy.optimize import minimize

# Parameters
TRADING_DAYS = 252
rf_annual = 0.01925

# Allocation constraints (top-down / inter-class weights)
W_ETF   = 0.40
W_SH    = 0.40
W_CRYP  = 0.20

# Volatility cap grid (same spirit as Part 1, but only vol_max)
vol_grid = np.arange(0.00, 1.00 + 1e-12, 0.01)
ridge = 1e-6  # covariance regularization

# Load price data
prices_ETF     = pd.read_excel(OUTPUT_DIR / "prices_and_correlation_ETF.xlsx", sheet_name="Prices_cleaned")
prices_shares  = pd.read_excel(OUTPUT_DIR / "prices_and_correlation_shares.xlsx", sheet_name="Prices_cleaned")
prices_cryptos = pd.read_excel(OUTPUT_DIR / "prices_and_correlation_cryptos.xlsx", sheet_name="Prices_cleaned")

def prep_prices(df):
    df = df.copy()
    # Some exports include a "Date" column
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.set_index("Date")
    df = df.select_dtypes(include=["number"])
    df = df.dropna(how="any")
    return df

prices_ETF     = prep_prices(prices_ETF)
prices_shares  = prep_prices(prices_shares)
prices_cryptos = prep_prices(prices_cryptos)

# Daily returns
ret_ETF     = prices_ETF.pct_change().dropna()
ret_shares  = prices_shares.pct_change().dropna()
ret_cryptos = prices_cryptos.pct_change().dropna()

# Sharpe ratio optimization under a volatility constraint (INTRA-CLASS)
def optimize_intra_class(returns_daily, vol_max, rf_annual, trading_days=252, ridge=1e-6):
    """
    Maximize Sharpe ratio under constraints:
      - sum(w) = 1
      - w_i >= 0 (long-only)
      - vol(w) <= vol_max

    Returns:
        dict(success, w_opt, sharpe, ret, vol, message, assets).
    """
    returns_daily = returns_daily.copy()

    mu = returns_daily.mean() * trading_days
    cov = returns_daily.cov() * trading_days

    assets = returns_daily.columns.tolist()
    n = len(assets)

    cov_mat = cov.values + ridge * np.eye(n)

    def port_return(w):
        return float(np.dot(w, mu.values))

    def port_vol(w):
        return float(np.sqrt(np.dot(w, cov_mat @ w)))

    def neg_sharpe(w):
        v = port_vol(w)
        if v <= 1e-12:
            return 1e6
        return -((port_return(w) - rf_annual) / v)

    constraints = [
        {"type": "eq",   "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: vol_max - port_vol(w)}
    ]
    bounds = [(0.0, 1.0)] * n
    w0 = np.ones(n) / n

    res = minimize(
        neg_sharpe,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 2000, "ftol": 1e-10, "disp": False}
    )

    out = {
        "success": bool(res.success),
        "message": str(res.message),
        "assets": assets,
        "w_opt": None,
        "ret": np.nan,
        "vol": np.nan,
        "sharpe": np.nan,
    }

    if res.success:
        w = res.x
        r = port_return(w)
        v = port_vol(w)
        s = (r - rf_annual) / v if v > 1e-12 else np.nan
        out.update({"w_opt": w, "ret": r, "vol": v, "sharpe": s})

    return out

def scan_vol_grid(returns_daily, vol_grid, rf_annual):
    """
    Scan vol_max values and keep the one that delivers the maximum Sharpe ratio.
    In case of ties, select the lowest vol_max (more conservative choice).

    Returns:
        (df_scan, best_vol_max) where best_vol_max is a float or None if no solution.
    """
    rows = []
    best = None

    for vol_max in vol_grid:
        out = optimize_intra_class(
            returns_daily,
            vol_max,
            rf_annual,
            trading_days=TRADING_DAYS,
            ridge=ridge
        )
        rows.append({
            "vol_max": vol_max,
            "success": out["success"],
            "message": out["message"],
            "sharpe": out["sharpe"],
            "ret": out["ret"],
            "vol": out["vol"],
            "n_active": int(np.sum(out["w_opt"] > 1e-4)) if out["success"] and out["w_opt"] is not None else np.nan
        })

    df = pd.DataFrame(rows)
    ok = df[df["success"]].copy()
    if ok.empty:
        return df, None

    best_sh = ok["sharpe"].max()
    candidates = ok[np.isclose(ok["sharpe"], best_sh)]
    best_row = candidates.sort_values(["vol_max"], ascending=True).iloc[0]
    return df, float(best_row["vol_max"])

In [6]:
# Takes about 6 minutes to run

# Scan vol_max by asset class
grid_ETF,     best_vol_ETF     = scan_vol_grid(ret_ETF, vol_grid, rf_annual)
grid_shares,  best_vol_shares  = scan_vol_grid(ret_shares, vol_grid, rf_annual)
grid_cryptos, best_vol_cryptos = scan_vol_grid(ret_cryptos, vol_grid, rf_annual)

print("Best vol_max ETF    :", best_vol_ETF)
print("Best vol_max Shares :", best_vol_shares)
print("Best vol_max Cryptos:", best_vol_cryptos)

# Final intra-class optimization
out_ETF     = optimize_intra_class(ret_ETF,     best_vol_ETF,     rf_annual, trading_days=TRADING_DAYS, ridge=ridge)
out_shares  = optimize_intra_class(ret_shares,  best_vol_shares,  rf_annual, trading_days=TRADING_DAYS, ridge=ridge)
out_cryptos = optimize_intra_class(ret_cryptos, best_vol_cryptos, rf_annual, trading_days=TRADING_DAYS, ridge=ridge)

# Safety check
if not (out_ETF["success"] and out_shares["success"] and out_cryptos["success"]):
    raise RuntimeError(
        f"Optimization failed for at least one asset class:\n"
        f"ETF: {out_ETF['message']}\n"
        f"Shares: {out_shares['message']}\n"
        f"Cryptos: {out_cryptos['message']}"
    )

# Final weights (40/40/20 × intra-class allocation)
w_ETF     = pd.Series(out_ETF["w_opt"],     index=out_ETF["assets"])     * W_ETF
w_shares  = pd.Series(out_shares["w_opt"],  index=out_shares["assets"])  * W_SH
w_cryptos = pd.Series(out_cryptos["w_opt"], index=out_cryptos["assets"]) * W_CRYP

w_final = pd.concat([w_ETF, w_shares, w_cryptos]).sort_values(ascending=False)
w_final = w_final[w_final > 1e-6]

print("\n--- Final weights (top-down + intra-class Sharpe optimization) ---")
print((w_final * 100).round(2))

Best vol_max ETF    : 0.04
Best vol_max Shares : 0.2
Best vol_max Cryptos: 0.75

--- Final weights (top-down + intra-class Sharpe optimization) ---
EHYB.MI      22.89
IBE.MC       21.03
BGRN         12.37
SOL-USD      10.24
ASML.AS       9.10
BTC-USD       5.99
NOVO-B.CO     5.31
SU.PA         4.56
EXH9.DE       3.51
MATIC-USD     2.51
HBAR-USD      1.26
PUST.PA       1.24
dtype: float64


### Final in-sample allocation obtained in the original run

| Asset | Weight |
|---|---:|
| EHYB.MI | 22.89% |
| IBE.MC | 21.03% |
| BGRN | 12.37% |
| SOL-USD | 10.24% |
| ASML.AS | 9.10% |
| BTC-USD | 5.99% |
| NOVO-B.CO | 5.31% |
| SU.PA | 4.56% |
| EXH9.DE | 3.51% |
| MATIC-USD | 2.51% |
| HBAR-USD | 1.26% |
| PUST.PA | 1.24% |

The original analysis selected volatility caps of **4% for ETFs**, **20% for equities**, and **75% for crypto-assets** before the final intra-class optimization.


In [8]:
final_assets = w_final.index.tolist()
prices_all_assets = pd.concat([prices_ETF, prices_shares, prices_cryptos], axis=1)
prices_final = prices_all_assets[final_assets]

prices_final = prices_final.dropna(how="any")
returns_final = prices_final.pct_change().dropna()

corr_matrix_final = returns_final.corr()

# Export Excel
file_name = OUTPUT_DIR / "final_portfolio_correlation.xlsx"
corr_matrix_final.to_excel(file_name)

# Optional automatic download in Google Colab
download_if_colab(file_name)
